# Fixing a Radial Build Model for fusion Applications

## Scenario
* Trying to model a Tokamak fusion power plant
* Have simple, but wrong, radial build model
* Perform parametric sweap

# Background
* Most promising fusion: deuterium ($^2H/D$) - tritium ($^3H/T$)

$$ D+T \to ^4He +n  + 17.6 {\rm MeV}$$
* Generally magnetic confinement is most viable for a fusion power plant
    * Such as a Tokamk 
 


## Example Tokamak

![ARC reactor concept](https://ars.els-cdn.com/content/image/1-s2.0-S0920379615302337-gr1.jpg)

ARC reactor concept from MIT. 
By: [Sorbom, et al.](https://dspace.mit.edu/handle/1721.1/112016), licensed under: [CC-BY-NC-ND-4.0](https://creativecommons.org/licenses/by-nc-nd/4.0/)

# The Need for Tritium Breeding
* Uses deuterium and tritium for fuel
* deuterium is abundant enough (see CANDUs)

## Tritium Pop Quiz
* What's the half-life of tritium?

* No natural source of tritium
* Inventory is constantly decaying
  * Bonus points for calculating the annual loss due to radioactive decay
* Current globla supply may supply a *single* power plant for a few years at most
* Must breed tritium onsite

$$ ^6Li + n \to T + ^4He$$

12.3 years

# Quick overview of Major Components Modeled
![exploded view of ARC reactor](https://ars.els-cdn.com/content/image/1-s2.0-S0920379615302337-gr3.jpg)

Exploded view of the ARC reactor. By: [Sorbom, et al.](https://dspace.mit.edu/handle/1721.1/112016), licensed under: [CC-BY-NC-ND-4.0](https://creativecommons.org/licenses/by-nc-nd/4.0/)

* Plasma
* First wall / armor
* Vacuum vessel
* Breeding blanket
* Neutron shielding and other in-vessel components (not modeled)
* Magnets

# Base Model

* 1-D radial build model
* Very simplified, and not representative
* Good enough for scoping
* Regions
  * Vacuum
  * First wall (tungsten)
  * Vacuum vessel ("stainless steel")
  * Breeder
  * void
  * Magnets (REBCO)

# Plot of Geometry

# First Steps
* Install MontePy
  * This is only needed for jupyter lite
  * Usually `pip install montepy` will persist (also on conda-forge)
* `import montepy`
* Read the model

In [ ]:
# Only needed on jupyter lite
from _config import install_montepy

install_montepy()

In [ ]:
import montepy

problem = montepy.read_input("models/fusion_tokomak.imcnp")

# Explore the model
* [See the doc strings](https://www.montepy.org/en/stable/api/generated/montepy.MCNP_Problem.html#montepy.MCNP_Problem)

In [ ]:
print(problem.cells)
print(problem.surfaces)
print(problem.materials)
print("version", problem.mcnp_version)

# Explore Cells
* Let's look at the cells and their comments

In [ ]:
for num, cell in problem.cells.items():
    #print the cell comments
    print(num, cell.comments)

# Update First Wall Cell
* grab the cell by its number (previous) step
* print the cell density
  * [helpful cell docs](https://www.montepy.org/en/stable/api/generated/montepy.Cell.html#montepy.Cell)
* print the material details
  * `repr(mat)` will be helpful

In [ ]:
first_wall = problem.cells[10]
print(first_wall.mass_density)

* This is near the theoretical maximum density for tungsten.
* First walls will likely not achieve this.
  * strong candidate is [tungsten-fiber reinforced tunsgten](https://doi.org/10.1088/1741-4326/ac8c55)
  * Can realistically only achieve around 80% density.

## task: Update mass density to be 80% theoretical 

In [ ]:
first_wall.mass_density *= 0.8

## Task 1.2 Check the material definition
* print out the composition of the material

In [ ]:
tungsten = first_wall.material
print(repr(tungsten))

# Update the Material definition
* tungsten has more than one stable isotope

## Steps
1. Make a dictionary of the stable isotopes using [wikipedia](https://en.wikipedia.org/wiki/Tungsten)
2. clear the material
3. add all nuclides to the material again
   1. [`add_nuclide`](https://www.montepy.org/en/stable/api/generated/montepy.Material.html#add-new-component)
   2. Use library: `82c`, ENDF/B-VII.1 at 900 K

In [ ]:
w_natural = {
    182: 0.265,
    183: 0.143,
    184: 0.306,
    186: 0.284
}
tungsten.clear()
for mass, abundance in w_natural.items():
    tungsten.add_nuclide(f"W-{mass}.82c", abundance)
print(repr(tungsten))

# Update material for Breeder blanket
* Inspect the material in use for the breeder

In [ ]:
breeder = problem.cells[15]
breeder_mat = breeder.material
print(repr(breeder_mat))

# Issue: Pure lithium metal is not a good breeder
* Alkali metals are annoying to work with
* Pure lithium is very low density (0.5 g/cc)
* Many options exist:
  * FLiBe
  * PbLi eutectic
  * $\rm Li_4SiO_4$

# Update to use FLiBe
* Starting with FLiBe
* Using [material mixing function](https://www.montepy.org/en/stable/api/generated/montepy.Materials.html#montepy.Materials.mix)
* Need to make elemental nuclides first
   * Bonus: why do we avoid atomic cross sections for neutron transport?
   * Have Li Already
   * Define [F](https://en.wikipedia.org/wiki/Fluorine) and [Be](https://en.wikipedia.org/wiki/Beryllium) materials
   * Use `82c` library again

In [ ]:
lithium = breeder_mat
flourine = montepy.Material()
flourine.add_nuclide("F-19.82c", 1.0)
beryllium = montepy.Material()
beryllium.add_nuclide("Be-9.82c", 1.0)

## Make the FLiBe
* Note: FLiBe infuriates chemists
* Actually mixture of $\rm LiF$ and $\rm BeF_2$
* Assume stochiametric mixture: $\rm Li_2[BeF_4]$
* Use ARC paper density: 1.94 g/cc

In [ ]:
flibe = problem.materials.mix([lithium, beryllium, flourine], [2, 1, 4])
flibe.normalize()
breeder.material = flibe
breeder.mass_density = 1.94
problem.materials.remove(lithium)

# Final task: Perform A Parametric Sweep
* Model has a lot of room for different thicknesses of tritium breeder blankets
* Make multiple models with different thicknesses (cylinder radii)

# Find the outer cylindrical surface first
* Each cell has a [`surfaces`](https://www.montepy.org/en/stable/api/generated/montepy.Cell.html#montepy.Cell.surfaces) attribute
* You can then filter by [surface type mnemonic](https://www.montepy.org/en/stable/api/generated/montepy.Surfaces.html#montepy.Surfaces.cz)

In [ ]:
cyls = list(breeder.surfaces.cz)
for cyl in cyls:
    print(cyl.number, cyl.radius)

In [ ]:
breeder_od = breeder.surfaces[20]

# Put it all together   
* use [`numpy.arange`](https://numpy.org/doc/stable/reference/generated/numpy.arange.html) to create array of radii
* Update the surfaces radius
* [`write_problem`](https://www.montepy.org/en/stable/api/generated/montepy.MCNP_Problem.html#montepy.MCNP_Problem.write_problem) to file

In [ ]:
import numpy as np
MIN_RADIUS = 115.01 # [cm]
MAX_RADIUS = 199.99 # [cm]

radii = np.arange(MIN_RADIUS, MAX_RADIUS, 5)
for radius in radii:
    breeder_od.radius = radius
    problem.write_problem(f"fusion_breeder_rad_{radius}.imcnp")

# Other Projects you can do
* Find common tritium breeder materials in the literature
  * Make all of them in the model, and test out different ones in the model
* Look into super conducting magnets
  * Change magnets from copper to high temp super conductors, e.g., [REBCO](https://doi.org/10.1098/rsta.2018.0354)
* Add tallies to the model
  * Neutron flux is always needed
  * Tritium production is essential
  * Nuclear heating is very important for coolant system sizing
  * Radiation damage (displacements per atom) is also important for keeping the structures and magnets working